In [2]:
!pip install transformers -q

In [3]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from transformers import AutoTokenizer, AutoModel
import gc

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [5]:
save_dir = "/home/ec2-user/SageMaker/embeddings/"
os.makedirs(save_dir, exist_ok=True)

In [6]:
train_df = pd.read_csv("/home/ec2-user/SageMaker/training_hhblits.csv")
test_df = pd.read_csv("/home/ec2-user/SageMaker/CB513.csv")

print(train_df.shape)  # (10792, 5)
print(test_df.shape)   # (511, 5)

(10792, 5)
(511, 5)


In [7]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
esm2 = AutoModel.from_pretrained("facebook/esm2_t6_8M_UR50D").to(device)
esm2.eval()

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
        

In [8]:
# Global variables
MAX_LEN = 700

In [9]:
def get_embedding(sequence, model, tokenizer, device, max_model_len=1022):
    # Truncate to model's max length (1022 + 2 special tokens = 1024)
    sequence = sequence[:max_model_len]
    tokens = tokenizer(sequence, return_tensors="pt").to(device)
    
    with torch.no_grad(): output = model(**tokens)
    
    # Remove [CLS] and [EOS] tokens, move to CPU immediately to free GPU memory
    emb = output.last_hidden_state[0, 1:-1, :].cpu().numpy()  # (seq_len, 320)
    return emb

In [12]:
def precompute_embeddings(df, model, tokenizer, device, save_dir, 
                          prefix, max_len=MAX_LEN, chunk_size=100):
    model.eval()
    os.makedirs(save_dir, exist_ok=True)
    
    chunk = []
    chunk_index  = 0

    for i, seq in enumerate(tqdm(df['input'], leave=False)):
        emb = get_embedding(seq, model, tokenizer, device)

        # Pad to MAX_LEN
        pad_len = max_len - len(emb)
        if pad_len > 0:
            emb = np.concatenate([emb, np.zeros((pad_len, emb.shape[1]))], axis=0)
        else:
            emb = emb[:max_len]  # truncate if somehow still too long

        chunk.append(emb.astype(np.float16))

        # Save chunk and clear RAM
        if len(chunk) == chunk_size:
            chunk_path = os.path.join(save_dir, f"{prefix}_chunk_{chunk_index:04d}.npy")
            np.save(chunk_path, np.array(chunk))
            print(f"Saved chunk {chunk_index} (proteins {chunk_index*chunk_size}–{i})")
            chunk       = []
            chunk_index += 1
            torch.cuda.empty_cache()
            gc.collect()

    # Save any remaining proteins
    if chunk:
        chunk_path = os.path.join(save_dir, f"{prefix}_chunk_{chunk_index:04d}.npy")
        np.save(chunk_path, np.array(chunk))
        # print(f"Saved final chunk {chunk_index}")

    # Combine all chunks into one file
    print("Combining chunks...")
    import glob
    chunks   = sorted(glob.glob(os.path.join(save_dir, f"{prefix}_chunk_*.npy")))
    combined = np.concatenate([np.load(c) for c in chunks], axis=0)
    
    final_path = os.path.join(save_dir, f"{prefix}_embeddings.npy")
    np.save(final_path, combined)
    print(f"Saved final embeddings: {combined.shape} → {final_path}")

    # Clean up chunk files
    for c in chunks:
        os.remove(c)

    return combined

In [13]:
train_embeddings = precompute_embeddings(train_df, esm2, tokenizer, device, save_dir, prefix="train")
test_embeddings = precompute_embeddings(test_df, esm2, tokenizer, device, save_dir, prefix="test")

  1%|          | 97/10792 [00:00<01:49, 97.96it/s] 

Saved chunk 0 (proteins 0–99)


  2%|▏         | 192/10792 [00:02<01:50, 96.15it/s]

Saved chunk 1 (proteins 100–199)


  3%|▎         | 294/10792 [00:03<01:51, 94.03it/s]

...

 78%|███████▊  | 400/511 [00:05<00:01, 59.05it/s]

Saved chunk 3 (proteins 300–399)


 96%|█████████▋| 493/511 [00:06<00:00, 95.12it/s]

Saved chunk 4 (proteins 400–499)


Combining chunks...
Saved final embeddings: (511, 700, 320) → /home/ec2-user/SageMaker/embeddings/test_embeddings.npy


In [14]:
import subprocess
subprocess.run(["aws", "s3", "cp", save_dir, "s3://your-bucket-name/embeddings/", "--recursive"])

Completed 218.3 MiB/4.7 GiB (0 Bytes/s) with 1 file(s) remaining

upload failed: embeddings/test_embeddings.npy to s3://your-bucket-name/embeddings/test_embeddings.npy An error occurred (AccessDenied) when calling the CreateMultipartUpload operation: Access Denied
upload failed: embeddings/train_embeddings.npy to s3://your-bucket-name/embeddings/train_embeddings.npy An error occurred (AccessDenied) when calling the CreateMultipartUpload operation: Access Denied


CompletedProcess(args=['aws', 's3', 'cp', '/home/ec2-user/SageMaker/embeddings/', 's3://your-bucket-name/embeddings/', '--recursive'], returncode=1)